In [ ]:
# ============================================================
# 1- Database design in phpMyAdmin -> SQL
# ============================================================

In [ ]:
# ============================================================
# 2- Import_runs table -> SQL
# ============================================================

In [1]:
# ============================================================
# 3- Phone Number Normalization
# ============================================================

import re

_PERSIAN_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)

_ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩",
    "0123456789"
)


def normalize_iran_phone(value) -> str:
    """
    Return an Iranian mobile number in local 10-digit form.

    Example:
        9121234567

    Accepted formats:
        9121234567
        09121234567
        +989121234567
        00989121234567

    Persian and Arabic-Indic digits are also accepted.
    """

    if value is None:
        raise ValueError("Phone number is empty")

    # Normalize Unicode digits first.
    s = str(value).strip()
    s = s.translate(_PERSIAN_DIGITS)
    s = s.translate(_ARABIC_DIGITS)

    # Reject obvious Excel numeric artifacts.
    if re.fullmatch(r"\d+\.0+", s):
        raise ValueError(
            f"Phone number appears to have been converted to a numeric "
            f"Excel value: {value!r}"
        )

    # Keep digits and an optional leading '+'.
    s = re.sub(r"[^\d+]", "", s)

    # Convert international Iranian formats to local format.
    if s.startswith("+98"):
        s = s[3:]
    elif s.startswith("0098"):
        s = s[4:]
    elif s.startswith("98") and len(s) == 12:
        s = s[2:]
    elif s.startswith("0"):
        s = s[1:]

    # Final Iranian mobile validation.
    if not re.fullmatch(r"9\d{9}", s):
        raise ValueError(
            f"Invalid Iranian mobile number: {value!r}"
        )

    return s

In [2]:
# ============================================================
# 4- Import Excel Data into the Database
# ============================================================
from pathlib import Path

import pandas as pd
import mysql.connector

EXCEL_FILE = Path(
    "/home/saeed/repositories/contacts/import_eitaa/contacts-1-44.xlsx"
)


def get_db_connection():
    return mysql.connector.connect(
        host="127.0.0.1",
        port=3306,
        user="reza",
        password="123",
        database="eitaa_import",
        charset="utf8mb4",
    )


def import_excel():
    df = pd.read_excel(
        EXCEL_FILE,
        dtype={
            "Full Name": "string",
            "Phone Number": "string",
        },
    )

    expected = {"Full Name", "Phone Number"}

    if set(df.columns) != expected:
        raise ValueError(
            f"Expected columns {expected}, got {set(df.columns)}"
        )

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        sql = """
            INSERT INTO contacts (
                full_name,
                phone_original,
                phone_normalized,
                status
            )
            VALUES (%s, %s, %s, 'pending')
            ON DUPLICATE KEY UPDATE
                full_name = VALUES(full_name),
                phone_original = VALUES(phone_original)
        """

        processed = 0
        invalid_phones = 0
        empty_names = 0

        for _, row in df.iterrows():

            # ------------------------------------------------
            # Full name
            # ------------------------------------------------
            if pd.isna(row["Full Name"]):
                empty_names += 1
                continue

            full_name = str(row["Full Name"]).strip()

            if not full_name:
                empty_names += 1
                continue

            # ------------------------------------------------
            # Phone number
            # ------------------------------------------------
            if pd.isna(row["Phone Number"]):
                invalid_phones += 1
                print("Invalid phone number: empty Excel cell")
                continue

            phone_original = str(row["Phone Number"]).strip()

            try:
                phone_normalized = normalize_iran_phone(
                    phone_original
                )
            except ValueError as exc:
                invalid_phones += 1
                print(
                    f"Invalid phone number "
                    f"{phone_original!r}: {exc}"
                )
                continue

            # ------------------------------------------------
            # Insert/update database
            # ------------------------------------------------
            cursor.execute(
                sql,
                (
                    full_name,
                    phone_original,
                    phone_normalized,
                ),
            )

            processed += 1

        conn.commit()

        # ----------------------------------------------------
        # Verify database state after import
        # ----------------------------------------------------
        cursor.execute(
            """
            SELECT
                COUNT(*) AS total,
                SUM(status = 'pending') AS pending,
                SUM(status = 'processing') AS processing,
                SUM(status = 'success') AS success,
                SUM(status = 'skipped_existing') AS skipped_existing,
                SUM(status = 'failed') AS failed
            FROM contacts
            """
        )

        stats = cursor.fetchone()

        print()
        print("========== IMPORT RESULT ==========")
        print(f"Excel rows processed : {processed}")
        print(f"Invalid phones       : {invalid_phones}")
        print(f"Empty names          : {empty_names}")
        print()
        print(f"DB total             : {stats[0]}")
        print(f"DB pending           : {stats[1]}")
        print(f"DB processing        : {stats[2]}")
        print(f"DB success           : {stats[3]}")
        print(f"DB skipped_existing  : {stats[4]}")
        print(f"DB failed            : {stats[5]}")
        print("====================================")

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    import_excel()



========== IMPORT RESULT ==========
Excel rows processed : 74
Invalid phones       : 0
Empty names          : 0

DB total             : 74
DB pending           : 74
DB processing        : 0
DB success           : 0
DB skipped_existing  : 0
DB failed            : 0


In [1]:
# ============================================================
# 5- Chrome session persistence
# ============================================================
# Clean, isolated Selenium Chrome session

from pathlib import Path
from tempfile import mkdtemp
from typing import Optional
import shutil

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import WebDriver

# ChromeDriver

CHROMEDRIVER = (
        Path.cwd()
        / "drivers"
        / "chromedriver"
        / "chromedriver"
)

# Create a completely new temporary Chrome User Data Directory

TEMP_CHROME_PROFILE = Path(
    mkdtemp(prefix="selenium-chrome-")
)

# Close previous Selenium session, if one exists

previous_driver: Optional[WebDriver] = globals().get("driver")

if previous_driver is not None:
    try:
        previous_driver.quit()
        print("\nPrevious Selenium driver closed.")
    except Exception as e:
        print("\nPrevious Selenium driver could not be closed:")
        print(repr(e))

# Chrome options

options = Options()

options.add_argument(
    f"--user-data-dir={TEMP_CHROME_PROFILE}"
)

options.add_argument("--start-maximized")

#  Explicit manual ChromeDriver
#  Selenium Manager is not used for driver discovery.


service = Service(
    executable_path=str(CHROMEDRIVER)
)

# Create new isolated Chrome session


driver: WebDriver = webdriver.Chrome(
    service=service,
    options=options,
)

driver.set_page_load_timeout(60)


First run:

driver = create_driver()

driver.get("https://web.eitaa.com/")

Log into Eitaa manually.

Close the browser normally.

On subsequent executions Selenium will reuse that Chrome profile/session.

This is much more robust than trying to automate OTP authentication.

In [2]:
# ============================================================
# 6- Close Selenium and remove temporary Chrome profile
# ============================================================

if "driver" in globals() and driver is not None:
    try:
        driver.quit()
        print("Selenium Chrome session closed.")
    finally:
        driver = None

if "TEMP_CHROME_PROFILE" in globals():
    shutil.rmtree(
        TEMP_CHROME_PROFILE,
        ignore_errors=True
    )

    print(
        "Temporary Selenium Chrome profile removed."
    )

Selenium Chrome session closed.
Temporary Selenium Chrome profile removed.


In [2]:
# ============================================================
# 7- Selenium Selectors
# ============================================================
# To test that the selector is unique:
# allow pasting
# document.querySelectorAll(".btn-menu-item.tgico-user.rp").length

from selenium.webdriver.common.by import By


class Selectors:
    # 1. Hamburger menu
    HAMBURGER_MENU = (
        By.CSS_SELECTOR,
        ".btn-icon.btn-menu-toggle.rp.sidebar-tools-button.is-visible"
    )

    # 2. Contacts menu item
    CONTACTS_BUTTON = (
        By.XPATH,
        "//div[contains(@class, 'btn-menu-item') and .//span[normalize-space()='Contacts']]"
    )

    # 3. "+" button inside Contacts
    ADD_CONTACT_BUTTON = (
        By.CSS_SELECTOR,
        "button.tgico-add.is-visible"
    )

    # 4. First name field
    FIRST_NAME = (
        By.CSS_SELECTOR,
        'div.input-field:first-child > div.input-field-input[contenteditable="true"]'
    )

    # Phone number field
    PHONE = (
        By.CSS_SELECTOR,
        'div.input-field-phone > '
        'div.input-field-input[contenteditable="true"]'
    )

    # Add/Save button
    SAVE_BUTTON = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-primary.btn-color-primary"
    )

In [3]:
# ============================================================
# 8- Reliable waits
# ============================================================
# Avoid: time.sleep(3) as the primary synchronization mechanism.
# Use WebDriverWait.

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

WAIT_SECONDS = 30


def wait_visible(driver, locator):
    return WebDriverWait(
        driver,
        WAIT_SECONDS
    ).until(
        EC.visibility_of_element_located(locator)
    )


def wait_clickable(driver, locator):
    return WebDriverWait(
        driver,
        WAIT_SECONDS
    ).until(
        EC.element_to_be_clickable(locator)
    )


In [10]:
# ============================================================
# 9- Add Contact to Eitaa - Diagnostic Version
# ============================================================

def add_contact(driver, full_name, phone_normalized):
    """
    Add exactly one Eitaa contact.

    Diagnostic version:
    prints progress so we can identify where execution stops.
    """

    print("STEP 1: Starting add_contact()")
    print("Full name:", full_name)
    print("Phone:", phone_normalized)

    # --------------------------------------------------------
    # Check Contacts page
    # --------------------------------------------------------

    contacts_containers = driver.find_elements(
        By.CSS_SELECTOR,
        "#contacts-container"
    )

    print(
        "STEP 2: Contacts containers found:",
        len(contacts_containers)
    )

    if contacts_containers:
        contacts_container = contacts_containers[0]

        contacts_visible = (
            "active"
            in contacts_container.get_attribute("class").split()
        )
    else:
        contacts_container = None
        contacts_visible = False

    print(
        "STEP 3: Contacts page active:",
        contacts_visible
    )

    # --------------------------------------------------------
    # Open Contacts if necessary
    # --------------------------------------------------------

    if not contacts_visible:
        print("STEP 4: Opening hamburger menu...")

        wait_clickable(
            driver,
            Selectors.HAMBURGER_MENU
        ).click()

        print("STEP 5: Hamburger menu clicked.")

        wait_clickable(
            driver,
            Selectors.CONTACTS_BUTTON
        ).click()

        print("STEP 6: Contacts button clicked.")

    else:
        print("STEP 4: Already on Contacts page.")

    # --------------------------------------------------------
    # Add Contact
    # --------------------------------------------------------

    print("STEP 7: Waiting for Add Contact button...")

    add_btn = WebDriverWait(
        driver,
        WAIT_SECONDS
    ).until(
        EC.element_to_be_clickable(
            (
                By.CSS_SELECTOR,
                "button.btn-circle.btn-corner.is-visible.rp"
            )
        )
    )

    print("STEP 8: Add Contact button found.")

    add_btn.click()

    print("STEP 9: Add Contact button clicked.")

    # --------------------------------------------------------
    # First name
    # --------------------------------------------------------

    first_name = wait_visible(
        driver,
        Selectors.FIRST_NAME
    )

    print("STEP 10: First Name field found.")

    first_name.click()
    first_name.send_keys(full_name)

    print("STEP 11: Full name entered.")

    # --------------------------------------------------------
    # Phone
    # --------------------------------------------------------

    phone = wait_visible(
        driver,
        Selectors.PHONE
    )

    print("STEP 12: Phone field found.")

    phone.click()
    phone.send_keys(phone_normalized)

    print("STEP 13: Phone number entered.")

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    save = wait_clickable(
        driver,
        Selectors.SAVE_BUTTON
    )

    print("STEP 14: Save button found.")

    save.click()

    print("STEP 15: Save button clicked.")

    # --------------------------------------------------------
    # Wait for popup to close
    # --------------------------------------------------------

    WebDriverWait(
        driver,
        WAIT_SECONDS
    ).until(
        EC.invisibility_of_element_located(
            (By.CSS_SELECTOR, ".popup-create-contact.active")
        )
    )

    print("STEP 16: Contact popup closed.")
    print("STEP 17: add_contact() completed successfully.")

    return True


In [12]:
result = add_contact(
    driver,
    "Test contact",
    "9188626037"
)

print("Result:", result)

STEP 1: Starting add_contact()
Full name: Test contact
Phone: 9188626037
STEP 2: Contacts containers found: 0
STEP 3: Contacts page active: False
STEP 4: Opening hamburger menu...
STEP 5: Hamburger menu clicked.
STEP 6: Contacts button clicked.
STEP 7: Waiting for Add Contact button...
STEP 8: Add Contact button found.
STEP 9: Add Contact button clicked.
STEP 10: First Name field found.
STEP 11: Full name entered.
STEP 12: Phone field found.
STEP 13: Phone number entered.
STEP 14: Save button found.
STEP 15: Save button clicked.
STEP 16: Contact popup closed.
STEP 17: add_contact() completed successfully.
Result: True


In [13]:
import inspect

print(inspect.getsource(add_contact))

def add_contact(driver, full_name, phone_normalized):
    """
    Add exactly one Eitaa contact.

    Diagnostic version:
    prints progress so we can identify where execution stops.
    """

    print("STEP 1: Starting add_contact()")
    print("Full name:", full_name)
    print("Phone:", phone_normalized)

    # --------------------------------------------------------
    # Check Contacts page
    # --------------------------------------------------------

    contacts_containers = driver.find_elements(
        By.CSS_SELECTOR,
        "#contacts-container"
    )

    print(
        "STEP 2: Contacts containers found:",
        len(contacts_containers)
    )

    if contacts_containers:
        contacts_container = contacts_containers[0]

        contacts_visible = (
            "active"
            in contacts_container.get_attribute("class").split()
        )
    else:
        contacts_container = None
        contacts_visible = False

    print(
        "STEP 3: Contacts page act

In [6]:
# ============================================================
# 9- Add Contact to Eitaa
# ============================================================

def add_contact(driver, full_name, phone_normalized):
    """
    Add exactly one Eitaa contact.

    Full name intentionally goes entirely into
    the first-name field.
    """

    # If you are not already in Contacts, open Contacts.
    contacts_containers = driver.find_elements(
        By.CSS_SELECTOR,
        "#contacts-container"
    )

    if contacts_containers:
        contacts_container = contacts_containers[0]
    else:
        contacts_container = None

    contacts_visible = (
            contacts_container is not None
            and "active" in contacts_container.get_attribute("class").split()
    )

    if not contacts_visible:
        wait_clickable(
            driver,
            Selectors.HAMBURGER_MENU
        ).click()

        wait_clickable(
            driver,
            Selectors.CONTACTS_BUTTON
        ).click()

    # Add Contact
    add_btn = WebDriverWait(
        driver,
        WAIT_SECONDS
    ).until(
        EC.element_to_be_clickable(
            (
                By.CSS_SELECTOR,
                "button.btn-circle.btn-corner.is-visible.rp"
            )
        )
    )

    add_btn.click()

    # Form
    first_name = wait_visible(
        driver,
        Selectors.FIRST_NAME
    )

    first_name.click()
    first_name.send_keys(full_name)

    phone = wait_visible(
        driver,
        Selectors.PHONE
    )

    phone.click()
    phone.send_keys(phone_normalized)

    # Save
    save = wait_clickable(
        driver,
        Selectors.SAVE_BUTTON
    )

    save.click()

    # Wait until the popup closes.
    WebDriverWait(
        driver,
        WAIT_SECONDS
    ).until(
        EC.invisibility_of_element_located(
            (By.CSS_SELECTOR, ".popup-create-contact.active")
        )
    )

    return True


In [5]:
# ============================================================
# 9- Add Contact to Eitaa - old
# ============================================================

def add_contact(driver, full_name, phone_normalized):
    """
    Add exactly one Eitaa contact.

    Full name intentionally goes entirely into
    the first-name field.
    """

    # If you are not already in Contacts, sign in to Contacts.
    contacts_container = driver.find_element(
        By.CSS_SELECTOR,
        "#contacts-container"
    )

    if not contacts_container.is_displayed():
        wait_clickable(driver, Selectors.HAMBURGER_MENU).click()
        wait_clickable(driver, Selectors.CONTACTS_BUTTON).click()

    # Add Contact
    add_btn = WebDriverWait(driver, WAIT_SECONDS).until(
        EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "#contacts-container button.tgico-add")
        )
    )
    add_btn.click()

    # Form
    first_name = wait_visible(driver, Selectors.FIRST_NAME)
    first_name.click()
    first_name.send_keys(full_name)

    phone = wait_visible(driver, Selectors.PHONE)
    phone.click()
    phone.send_keys(phone_normalized)

    # Save
    save = wait_clickable(driver, Selectors.SAVE_BUTTON)
    save.click()

    # Wait until the popup closes.
    WebDriverWait(driver, WAIT_SECONDS).until(
        EC.invisibility_of_element_located(
            (By.CSS_SELECTOR, ".popup-create-contact.active")
        )
    )

    return True

In [7]:
# ============================================================
# 10- Fetching the Next Contact
# ============================================================
def get_next_contact(conn):
    cursor = conn.cursor(dictionary=True)

    try:
        cursor.execute("""
            SELECT
                id,
                full_name,
                phone_original,
                phone_normalized,
                attempts
            FROM contacts
            WHERE status IN ('pending', 'failed')
              AND attempts < 3
            ORDER BY id
            LIMIT 1
        """)

        return cursor.fetchone()

    finally:
        cursor.close()

# You may want failed contacts handled separately rather than automatically retried. For 5,000 records, I generally recommend:

# pending → automatic retry
# failed  → max 2-3 attempts

# and then review the failures manually.

In [8]:
# ============================================================
# 11- Complete processing loop
# ============================================================
# Here is the basic architecture I would use:

import logging
from datetime import datetime

from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
)

logging.basicConfig(
    filename="eitaa_import.log",
    level=logging.INFO,
    format=(
        "%(asctime)s | %(levelname)s | "
        "%(message)s"
    ),
)


def mark_processing(conn, contact_id):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET
                status = 'processing',
                attempts = attempts + 1,
                started_at = NOW()
            WHERE id = %s
        """, (contact_id,))

        conn.commit()

    finally:
        cursor.close()


def mark_success(conn, contact_id):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET
                status = 'success',
                error_message = NULL,
                completed_at = NOW()
            WHERE id = %s
        """, (contact_id,))

        conn.commit()

    finally:
        cursor.close()


def mark_failed(conn, contact_id, error):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET
                status = 'failed',
                error_message = %s
            WHERE id = %s
        """, (
            str(error)[:60000],
            contact_id,
        ))

        conn.commit()

    finally:
        cursor.close()


def reset_stale_processing(conn):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET status = 'pending'
            WHERE status = 'processing'
              AND started_at <
                  NOW() - INTERVAL 30 MINUTE
        """)

        conn.commit()

    finally:
        cursor.close()


def run_import():
    conn = get_db_connection()

    driver = None

    try:
        reset_stale_processing(conn)

        driver = create_driver()

        driver.get("https://web.eitaa.com/")

        # At this point, verify that the existing
        # Eitaa session is still authenticated.
        #
        # If not, stop and ask for manual login
        # rather than attempting OTP automatically.

        while True:

            contact = get_next_contact(conn)

            if contact is None:
                logging.info("No more contacts.")
                break

            contact_id = contact["id"]
            full_name = contact["full_name"]
            phone = contact["phone_normalized"]

            logging.info(
                "Processing id=%s name=%r phone=%s",
                contact_id,
                full_name,
                phone,
            )

            mark_processing(conn, contact_id)

            try:

                add_contact(
                    driver,
                    full_name,
                    phone,
                )

                mark_success(
                    conn,
                    contact_id,
                )

                logging.info(
                    "SUCCESS id=%s",
                    contact_id,
                )

            except TimeoutException as exc:

                mark_failed(
                    conn,
                    contact_id,
                    f"Timeout: {exc}",
                )

                logging.exception(
                    "TIMEOUT id=%s",
                    contact_id,
                )

            except WebDriverException as exc:

                mark_failed(
                    conn,
                    contact_id,
                    f"WebDriver error: {exc}",
                )

                logging.exception(
                    "WEBDRIVER ERROR id=%s",
                    contact_id,
                )

                # Browser may be unusable.
                # Recreate it.

                try:
                    driver.quit()
                except Exception:
                    pass

                driver = create_driver()
                driver.get(
                    "https://web.eitaa.com/"
                )

            except Exception as exc:

                mark_failed(
                    conn,
                    contact_id,
                    repr(exc),
                )

                logging.exception(
                    "ERROR id=%s",
                    contact_id,
                )

    finally:

        if driver is not None:
            try:
                driver.quit()
            except Exception:
                pass

        conn.close()

In [ ]:
# ============================================================
# Chrome crash recovery
# ============================================================
except WebDriverException:

logging.exception(
    "Chrome/WebDriver failure"
)

try:
    driver.quit()
except Exception:
    pass

driver = create_driver()

driver.get(
    "https://web.eitaa.com/"
)

# Wait for authenticated Eitaa state again.

In [ ]:
# ============================================================
# Add screenshots for failures
# ============================================================
from pathlib import Path
from datetime import datetime

SCREENSHOT_DIR = Path("screenshots")
SCREENSHOT_DIR.mkdir(exist_ok=True)


def save_failure_screenshot(driver, contact_id):
    filename = (
            SCREENSHOT_DIR
            / f"{contact_id}_{datetime.now():%Y%m%d_%H%M%S}.png"
    )

    driver.save_screenshot(str(filename))

    return filename




In [ ]:
except Exception as exc:

screenshot = save_failure_screenshot(
    driver,
    contact_id,
)

mark_failed(
    conn,
    contact_id,
    f"{exc}\nScreenshot: {screenshot}",
)